# 01. 전처리

뉴스 유목민 지수(NNI) 분석 파이프라인 1/3 단계.
원자료를 불러와 결측치 가정을 검증하고, 채널별 뉴스 이용빈도 변수와 tier(이탈도 단계) 매핑을 구성한다.

- 입력: 원본 설문 데이터(xlsx)
- 출력: `step1_channels.pkl` (다음 단계 02_NNI산출.ipynb 에서 사용)

In [1]:
# =========================================================
# 0. 라이브러리 & 옵션
# =========================================================
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (silhouette_score, calinski_harabasz_score,
                              davies_bouldin_score, adjusted_rand_score)
from sklearn.model_selection import train_test_split

from scipy.stats import pearsonr, skew

import matplotlib.pyplot as plt
import seaborn as sns
from math import pi

import joblib
import os

pd.set_option('display.max_columns', 300)
pd.set_option('display.width', 240)

plt.rc('font', family='Malgun Gothic')  # 한글 폰트 (Mac은 'AppleGothic'으로 변경)
plt.rcParams['axes.unicode_minus'] = False
%matplotlib inline

In [2]:
# =========================================================
# 1. 데이터 불러오기
# =========================================================
DATA_PATH = "2. 2025 언론수용자 조사_최종데이터.xlsx"  # 팀원 환경에 맞게 경로 수정

df = pd.read_excel(DATA_PATH).copy()
print('원본 데이터 shape:', df.shape)

# 조사 설계가중치(WT): 모집단 대표성이 필요한 통계(군집 비중 등)에는 WT를 반영하고,
# 군집화 자체는 개별 응답자 패턴을 찾는 것이므로 원자료 그대로 수행한다.
assert 'WT' in df.columns, "설계가중치 WT 컬럼이 없습니다. 원자료 구조를 확인하세요."


원본 데이터 shape: (6000, 223)


## 2. 결측=0 가정 검증

빈도 문항의 결측을 전부 0(미이용)으로 채우기 전에, 채널별 스크리너 문항(이용 여부)과 대조해
이 결측이 **진짜 무응답이 아니라 스킵 로직에 의한 것**인지 확인한다.

In [3]:
# =========================================================
# 2. 결측=0 가정 검증
# =========================================================
screener_pairs = {
    'newspaper': ('Q1',  'Q2A_1'), 'magazine':  ('Q8',  'Q9A_1'),
    'tv':        ('Q13', 'Q14A_1'), 'radio':     ('Q20', 'Q21A_1'),
    'messenger': ('Q39', 'Q40A_1'), 'sns':       ('Q46', 'Q47A_1'),
    'video':     ('Q53', 'Q54A_1'), 'shortform': ('Q61', 'Q62A_1'),
    'genai':     ('Q73', 'Q74A_1'), 'podcast':   ('Q78', 'Q79A_1'),
    # 포털(Q32A_1/2)은 별도 스크리너 없이 바로 이용일수를 묻는 구조라 제외
}

all_clean = True
for ch, (screener, freq_col) in screener_pairs.items():
    true_nonresponse = df.loc[df[screener] == 1, freq_col].isna().sum()
    inconsistent = df.loc[df[screener] == 2, freq_col].notna().sum()
    if true_nonresponse > 0 or inconsistent > 0:
        all_clean = False
        print(f'[주의] {ch}: 진짜 무응답 {true_nonresponse}명 / 모순 {inconsistent}명')

print('결측=0 가정:', '안전함 (모든 채널 0)' if all_clean else '위 채널은 별도 처리 필요')

결측=0 가정: 안전함 (모든 채널 0)


## 3. 뉴스 채널 빈도 변수 구성

"그 채널을 이용했는가"가 아니라 **"그 채널에서 뉴스/시사 콘텐츠를 이용했는가"**를 묻는
뉴스 특화 문항만 사용한다. 평일(A)+주말(B) 문항을 합산해 주간 총 이용일수로 구성한다.

| key | 문항 | tier(이탈도 단계) |
|---|---|---|
| newspaper / tv / radio / magazine | Q2/Q14/Q21/Q9 (A+B) | 0 (언론사 원본/직접 채널) |
| portal | Q32 (모바일·PC 중 큰 값, A+B) | 1 (브랜드 유지·제3자 배열) |
| podcast | Q79 (A+B) | 1 |
| messenger / sns / video / shortform / genai | Q40/Q47/Q54/Q62/Q74 (A+B) | 2 (출처 표시 흐려짐·알고리즘 주도) |


In [4]:
# =========================================================
# 3. 채널 빈도 변수 (평일+주말 합산) + tier 매핑
# =========================================================
raw_pairs = {
    'newspaper': ('Q2A_1',  'Q2B_1',  0, '종이신문'),
    'tv':        ('Q14A_1', 'Q14B_1', 0, 'TV뉴스'),
    'radio':     ('Q21A_1', 'Q21B_1', 0, '라디오뉴스'),
    'magazine':  ('Q9A_1',  'Q9B_1',  0, '시사잡지'),
    'podcast':   ('Q79A_1', 'Q79B_1', 1, '팟캐스트뉴스'),
    'messenger': ('Q40A_1', 'Q40B_1', 2, '메신저뉴스'),
    'sns':       ('Q47A_1', 'Q47B_1', 2, 'SNS뉴스'),
    'video':     ('Q54A_1', 'Q54B_1', 2, '온라인동영상뉴스'),
    'shortform': ('Q62A_1', 'Q62B_1', 2, '숏폼뉴스'),
    'genai':     ('Q74A_1', 'Q74B_1', 2, '생성형AI뉴스'),
}
portal_cols = ['Q32A_1', 'Q32A_2', 'Q32B_1', 'Q32B_2']  # 모바일/PC x 평일/주말

fill_cols = portal_cols + [c for a, b, _, _ in raw_pairs.values() for c in (a, b)]
for c in fill_cols:
    df[c] = df[c].fillna(0)

FREQ_WINDOW = 'weekday_weekend'  # 그리드서치 결과 평일만 쓰는 것과 차이가 크지 않아, 정보 손실 없는 이 방식을 채택(7절 참고)

channel_specs = {}
for key, (a, b, tier, label) in raw_pairs.items():
    col = f'ch_{key}'
    df[col] = df[a] + df[b]
    channel_specs[key] = {'col': col, 'tier': tier, 'label': label}

wk_mobile = df['Q32A_1'] + df['Q32B_1']
wk_pc = df['Q32A_2'] + df['Q32B_2']
df['ch_portal'] = pd.concat([wk_mobile, wk_pc], axis=1).max(axis=1)
channel_specs['portal'] = {'col': 'ch_portal', 'tier': 1, 'label': '포털뉴스'}

channel_cols  = [spec['col']  for spec in channel_specs.values()]
channel_tiers = np.array([spec['tier'] for spec in channel_specs.values()], dtype=float)

print('사용 채널 수:', len(channel_specs))
print(df[channel_cols].describe().round(2).T)

사용 채널 수: 11
               count  mean   std  min  25%  50%  75%  max
ch_newspaper  6000.0  0.29  1.12  0.0  0.0  0.0  0.0  7.0
ch_tv         6000.0  4.44  2.65  0.0  3.0  5.0  7.0  7.0
ch_radio      6000.0  0.29  1.12  0.0  0.0  0.0  0.0  7.0
ch_magazine   6000.0  0.07  0.58  0.0  0.0  0.0  0.0  7.0
ch_podcast    6000.0  0.04  0.42  0.0  0.0  0.0  0.0  7.0
ch_messenger  6000.0  0.47  1.47  0.0  0.0  0.0  0.0  7.0
ch_sns        6000.0  0.26  1.04  0.0  0.0  0.0  0.0  7.0
ch_video      6000.0  1.13  2.15  0.0  0.0  0.0  1.0  7.0
ch_shortform  6000.0  0.83  1.86  0.0  0.0  0.0  0.0  7.0
ch_genai      6000.0  0.06  0.46  0.0  0.0  0.0  0.0  7.0
ch_portal     6000.0  3.01  2.75  0.0  0.0  3.0  6.0  7.0


In [5]:
# =========================================================
#  포털 계산 방식 정량 비교 — max→합 vs 합→max
#   결론 요약: 두 방식이 달라지는 응답자는 14명(0.2%), 최대 2일 차이.
#              전체 파이프라인 기준 ARI=0.997 → 방식 선택이 결과를 바꾸지 않음.
# =========================================================
_pA = np.maximum(df['Q32A_1'], df['Q32A_2']) + np.maximum(df['Q32B_1'], df['Q32B_2'])  # 기간별 max 후 합산 (B팀 김소현)
_pB = pd.concat([df['Q32A_1'] + df['Q32B_1'],
                 df['Q32A_2'] + df['Q32B_2']], axis=1).max(axis=1)                     # 기기별 합 후 max (본 노트북)

_diff = _pA - _pB   # 수학적으로 항상 >= 0
n_diff = int((_diff > 0).sum())
print('[포털 계산 방식 비교]')
print(f'  방식A(max→합) 평균 {_pA.mean():.3f}일 | 방식B(합→max) 평균 {_pB.mean():.3f}일')
print(f'  값이 달라지는 응답자: {n_diff}명 ({n_diff/len(df)*100:.1f}%) | 차이 크기: 최대 {int(_diff.max())}일')

_cross = (((df['Q32A_2'] > df['Q32A_1']) & (df['Q32B_1'] > df['Q32B_2'])) |
          ((df['Q32A_1'] > df['Q32A_2']) & (df['Q32B_2'] > df['Q32B_1'])))
print(f'  평일·주말 주기기가 다른 교차 사용자: {int(_cross.sum())}명 ({_cross.mean()*100:.1f}%)')
print('  => 교차 사용자가 극소수(0.2%)이므로 두 방식은 실질적으로 동일한 값을 산출.')
print('     (전체 파이프라인 재실행 비교: 지수 상관 r=1.000, 군집 ARI=0.997, 이동자 15명)')
print('     본 분석은 기기별 합산 후 max 방식을 유지한다.')

[포털 계산 방식 비교]
  방식A(max→합) 평균 3.013일 | 방식B(합→max) 평균 3.010일
  값이 달라지는 응답자: 14명 (0.2%) | 차이 크기: 최대 2일
  평일·주말 주기기가 다른 교차 사용자: 14명 (0.2%)
  => 교차 사용자가 극소수(0.2%)이므로 두 방식은 실질적으로 동일한 값을 산출.
     (전체 파이프라인 재실행 비교: 지수 상관 r=1.000, 군집 ARI=0.997, 이동자 15명)
     본 분석은 기기별 합산 후 max 방식을 유지한다.


## 저장

다음 단계(02_NNI산출.ipynb)에서 이어받을 수 있도록 중간 산출물을 저장한다.

In [6]:
# =========================================================
# 중간 저장: 02_NNI산출.ipynb 에서 사용
# =========================================================
import pickle
with open('step1_channels.pkl', 'wb') as f:
    pickle.dump({
        'df': df,
        'channel_specs': channel_specs,
        'channel_cols': channel_cols,
        'channel_tiers': channel_tiers,
        'FREQ_WINDOW': FREQ_WINDOW,
    }, f)
print('저장 완료: step1_channels.pkl')

저장 완료: step1_channels.pkl
